# Seed Azure SQL Database

Loads the retail CSVs from the lakehouse into the provisioned **Azure SQL Database**
so it can play the role of an operational POS/ERP system.

Tables are created with **primary keys** — a prerequisite for Fabric Database
Mirroring (tables without a PK are not replicated).

Authentication is **Microsoft Entra-only** (no SQL password): this notebook runs as
you — the server's Entra admin — and also grants the Fabric **workspace identity**
the permissions Mirroring needs to read the source database.

**Reads:** `{{DATA_SOURCE_PATH}}` CSVs  **Writes:** `dbo.products`, `dbo.stores`, `dbo.pos_transactions`, `dbo.inventory_snapshots`

In [ ]:
SQL_SERVER = "{{SQL_SERVER}}"        # e.g. myserver.database.windows.net
SQL_DATABASE = "{{SQL_DATABASE}}"
WORKSPACE_IDENTITY_NAME = "{{WORKSPACE_IDENTITY_NAME}}"  # Fabric workspace identity (= workspace name)

# Microsoft Entra-only authentication — no SQL login/password exists on this server.
# This notebook runs as you (the server's Entra admin); acquire a SQL access token.
import notebookutils

def _sql_access_token():
    errors = []
    for audience in ("https://database.windows.net/", "https://database.windows.net"):
        try:
            tok = notebookutils.credentials.getToken(audience)
            if tok:
                return tok
        except Exception as e:  # noqa: BLE001
            errors.append(f"{audience}: {e}")
    raise RuntimeError("Could not acquire an Azure SQL access token via "
                       "notebookutils.credentials.getToken — " + "; ".join(errors))

ACCESS_TOKEN = _sql_access_token()

JDBC_URL = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;database={SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;loginTimeout=60;"
)
JDBC_PROPS = {
    "accessToken": ACCESS_TOKEN,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

def sql_connection(database=None):
    """Open a raw JDBC connection using the Entra access token."""
    url = (f"jdbc:sqlserver://{SQL_SERVER}:1433;database={database or SQL_DATABASE};"
           "encrypt=true;trustServerCertificate=false;loginTimeout=60;")
    jvm = spark._sc._jvm
    props = jvm.java.util.Properties()
    props.setProperty("accessToken", ACCESS_TOKEN)
    jvm.java.lang.Class.forName("com.microsoft.sqlserver.jdbc.SQLServerDriver")
    return jvm.java.sql.DriverManager.getConnection(url, props)

print(f"Target: {SQL_SERVER}/{SQL_DATABASE} (Microsoft Entra auth)")

In [ ]:
# Read the staged CSVs from the lakehouse
def read_csv(name):
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(f"{{DATA_SOURCE_PATH}}/{name}.csv")
    )
    print(f"{name}: {df.count():,} rows")
    return df

products_df = read_csv("products")
stores_df = read_csv("stores")
txn_df = read_csv("pos_transactions")
inv_df = read_csv("inventory_snapshots")

In [ ]:
# Create tables with PRIMARY KEYS via raw JDBC (Spark's own writer cannot add PKs,
# and Fabric Mirroring only replicates tables that have one).
DDL = """
IF OBJECT_ID('dbo.pos_transactions','U') IS NOT NULL DROP TABLE dbo.pos_transactions;
IF OBJECT_ID('dbo.inventory_snapshots','U') IS NOT NULL DROP TABLE dbo.inventory_snapshots;
IF OBJECT_ID('dbo.products','U') IS NOT NULL DROP TABLE dbo.products;
IF OBJECT_ID('dbo.stores','U') IS NOT NULL DROP TABLE dbo.stores;

CREATE TABLE dbo.products (
    sku          VARCHAR(20)   NOT NULL PRIMARY KEY,
    product_name NVARCHAR(200) NULL,
    category     NVARCHAR(50)  NULL,
    subcategory  NVARCHAR(50)  NULL,
    brand        NVARCHAR(50)  NULL,
    unit_cost    DECIMAL(10,2) NULL
);
CREATE TABLE dbo.stores (
    store_id     VARCHAR(20)   NOT NULL PRIMARY KEY,
    store_name   NVARCHAR(100) NULL,
    city         NVARCHAR(50)  NULL,
    state        NVARCHAR(20)  NULL,
    region       NVARCHAR(20)  NULL,
    store_format NVARCHAR(30)  NULL
);
CREATE TABLE dbo.pos_transactions (
    transaction_id        VARCHAR(30)   NOT NULL PRIMARY KEY,
    store_id              VARCHAR(20)   NULL,
    product_id            VARCHAR(20)   NULL,
    transaction_timestamp DATETIME2     NULL,
    quantity              INT           NULL,
    unit_price            DECIMAL(10,2) NULL,
    discount_pct          DECIMAL(5,2)  NULL
);
CREATE TABLE dbo.inventory_snapshots (
    snapshot_date     DATE        NOT NULL,
    store_id          VARCHAR(20) NOT NULL,
    product_id        VARCHAR(20) NOT NULL,
    quantity_on_hand  INT         NULL,
    quantity_on_order INT         NULL,
    reorder_point     INT         NULL,
    CONSTRAINT pk_inventory PRIMARY KEY (snapshot_date, store_id, product_id)
);
"""

conn = sql_connection()
stmt = conn.createStatement()
for batch in [b.strip() for b in DDL.split(";") if b.strip()]:
    stmt.execute(batch)
stmt.close()
conn.close()
print("Tables created with primary keys")

In [ ]:
# Grant the Fabric workspace identity the permissions Mirroring requires.
# Runs as you (the Entra admin). The workspace identity is a contained Entra
# user in the mirrored database — no SQL login or password is involved.
import time

wi = WORKSPACE_IDENTITY_NAME.replace("]", "]]")  # escape ] for bracket-quoting

def _exec(conn, sql):
    stmt = conn.createStatement()
    try:
        stmt.execute(sql)
    finally:
        stmt.close()

# Create the contained user (retry: the workspace identity can take a minute
# to propagate in Microsoft Entra after provisioning).
create_user_sql = (
    f"IF NOT EXISTS (SELECT 1 FROM sys.database_principals WHERE name = N'{WORKSPACE_IDENTITY_NAME}') "
    f"CREATE USER [{wi}] FROM EXTERNAL PROVIDER;"
)
last_err = None
for attempt in range(10):
    conn = sql_connection()
    try:
        _exec(conn, create_user_sql)
        last_err = None
        break
    except Exception as e:  # noqa: BLE001
        last_err = e
        print(f"[{attempt+1}/10] workspace identity not resolvable yet: {str(e)[:160]}")
        time.sleep(15)
    finally:
        conn.close()
if last_err is not None:
    raise last_err

# Grant the minimum permissions Fabric Mirroring needs on the source database.
conn = sql_connection()
try:
    for perm in ("SELECT", "ALTER ANY EXTERNAL MIRROR",
                 "VIEW DATABASE PERFORMANCE STATE", "VIEW DATABASE SECURITY STATE"):
        _exec(conn, f"GRANT {perm} TO [{wi}];")
finally:
    conn.close()

print(f"Workspace identity '{WORKSPACE_IDENTITY_NAME}' granted mirroring permissions")

In [ ]:
# Append data through Spark JDBC (tables already exist with the right schema)
from pyspark.sql.functions import col, to_date, to_timestamp

(products_df
 .select("sku", "product_name", "category", "subcategory", "brand",
         col("unit_cost").cast("decimal(10,2)"))
 .write.mode("append").jdbc(JDBC_URL, "dbo.products", properties=JDBC_PROPS))
print("products loaded")

(stores_df
 .select("store_id", "store_name", "city", "state", "region", "store_format")
 .write.mode("append").jdbc(JDBC_URL, "dbo.stores", properties=JDBC_PROPS))
print("stores loaded")

(txn_df
 .select("transaction_id", "store_id", "product_id",
         to_timestamp("transaction_timestamp").alias("transaction_timestamp"),
         col("quantity").cast("int"),
         col("unit_price").cast("decimal(10,2)"),
         col("discount_pct").cast("decimal(5,2)"))
 .write.mode("append").jdbc(JDBC_URL, "dbo.pos_transactions", properties=JDBC_PROPS))
print("pos_transactions loaded")

(inv_df
 .select(to_date("snapshot_date").alias("snapshot_date"), "store_id", "product_id",
         col("quantity_on_hand").cast("int"),
         col("quantity_on_order").cast("int"),
         col("reorder_point").cast("int"))
 .dropDuplicates(["snapshot_date", "store_id", "product_id"])
 .write.mode("append").jdbc(JDBC_URL, "dbo.inventory_snapshots", properties=JDBC_PROPS))
print("inventory_snapshots loaded")

In [ ]:
# Verify row counts straight from SQL
for table in ["products", "stores", "pos_transactions", "inventory_snapshots"]:
    n = (spark.read.jdbc(JDBC_URL, f"(SELECT COUNT(*) AS n FROM dbo.{table}) q",
                         properties=JDBC_PROPS).collect()[0]["n"])
    print(f"dbo.{table}: {n:,} rows")
print("\nAzure SQL database seeded — ready for mirroring")